In [1]:
print("hello world")

hello world


## PDF Q&A System (RAG) – First Step

#### Goal: Upload a PDF, ask questions, get answers with source citations.

In [1]:
# pip install langchain langchain-community langchain-openai pypdf chromadb tiktoken
# pip install -qU langchain-huggingface sentence-transformers

In [2]:
import langchain
print(f"langchain version: {langchain.__version__}")

langchain
print(f"langchain.chains location: {langchain.__file__}")

langchain version: 1.2.17
langchain.chains location: /opt/anaconda3/envs/ai_dev/lib/python3.12/site-packages/langchain/__init__.py


In [3]:
# Uninstall any existing langchain packages
# !pip uninstall -y langchain langchain-core langchain-community

# Install the standard v0.2/v0.3 compatible packages
# !pip install langchain langchain-core langchain-community

In [4]:
import chromadb
import pypdf
import tiktoken

print(f"chromedb version: {chromadb.__version__}")
print(f"pypdf version: {pypdf.__version__}")
print(f"tiktoken version: {tiktoken.__version__}")

chromedb version: 1.5.8
pypdf version: 6.10.2
tiktoken version: 0.12.0


1. import libraries

In [5]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
# from langchain.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate


2. load pdf

In [6]:
pdf_filename = "e-brochure-uc-taisor.pdf"
pdf_loader = PyPDFLoader(pdf_filename)
pdf_documents = pdf_loader.load()
print(f"Loaded {len(pdf_documents)} documents from {pdf_filename}")


Loaded 24 documents from e-brochure-uc-taisor.pdf


3. split into chunks

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(pdf_documents)
# print(chunks[0].metadata)  # Should show {'page': 0, 'source': 'sample.pdf'}
for i, chunk in enumerate(chunks):
    chunk.metadata['page'] = chunk.metadata.get('page', i)  # fallback
print(chunks[5].metadata) 

{'producer': 'Adobe PDF library 17.00', 'creator': 'Adobe Illustrator 29.7 (Windows)', 'creationdate': '2025-08-12T16:04:30+06:30', 'moddate': '2025-08-12T16:04:30+05:30', 'title': 'UC Taisor_Mobile Brochure', 'source': 'e-brochure-uc-taisor.pdf', 'total_pages': 24, 'page': 5, 'page_label': '6'}


4. create vector store (embedding + storage)

In [8]:
# import os
# from dotenv import load_dotenv, find_dotenv
# load_dotenv(find_dotenv())

# az_aoai_key = os.getenv("az_aoai_key")
# az_model = os.getenv("az_model")
# az_endpoint = os.getenv("az_endpoint")
# api_ver = os.getenv("az_api_ver")

from langchain_huggingface import HuggingFaceEmbeddings


embedd = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embedding=embedd)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

5. create a model and retrieval QA chain

In [9]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

az_aoai_key = os.getenv("az_aoai_key")
az_model = os.getenv("az_model")
az_endpoint = os.getenv("az_endpoint")
api_ver = os.getenv("api_ver")



chat_model = AzureChatOpenAI(
    api_key=az_aoai_key,  # type: ignore
    api_version=api_ver,
    model=az_model,
    azure_endpoint=az_endpoint,
    temperature=0.01
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate

sys_prompt = ChatPromptTemplate.from_messages([ # type: ignore
    ("system", """You are a helpful assistant that answers questions based on the provided context.
    Always cite the source page number at the end of each sentence or paragraph when possible.
    If the answer is not in the context, say "I don't know" and do not make up information.
    
    Context:
    {context}"""),
    ("human", "{input}")
])

combine_docs_chain = create_stuff_documents_chain(llm=chat_model, prompt=sys_prompt)
retrieve_docs_chain = vector_store.as_retriever(search_kwargs={"k": 3})
rag_qna_chain = create_retrieval_chain(retrieve_docs_chain, combine_docs_chain) # type: ignore

6. execute the chain to answer user queries

In [11]:
while True:
    user_query = input("Enter your question (or 'exit' to quit): ")
    if user_query.lower() in ['exit', 'quit', 'bye']:
        print("Exiting the program. Goodbye!")
        break
    results = rag_qna_chain.invoke({"input": user_query}) # type: ignore
    print(f"\n\tBot Answer: {results['answer']}") # type: ignore
    # print(f"\n\tSource Documents: {retrieve_docs_chain.get_relevant_documents(user_query)}") # type: ignore
    print("\nSources:")
    for doc in results['context']:
        page = doc.metadata.get('page', 'unknown')
        source = doc.metadata.get('source', 'unknown')
        print(f"- {source}, page {page}: {doc.page_content[:100]}...")

Exiting the program. Goodbye!


## 🎯 Key Takeaways from Project 2 (PDF Q&A with RAG)

1. **RAG is a four-step pipeline**  
   - **Load** → documents from PDF (or any source)  
   - **Split** → into semantic chunks (size matters)  
   - **Embed** → convert chunks to vectors  
   - **Retrieve** → find relevant chunks for a query  

2. **Vector stores are search engines for meaning**  
   - Chroma (or FAISS, Pinecone) stores embeddings and enables similarity search.  
   - The retriever (`vectorstore.as_retriever()`) is the interface for fetching relevant chunks.

3. **Modern RAG uses two composable chains**  
   - `create_stuff_documents_chain` (LLM + prompt that receives `context` and `input`)  
   - `create_retrieval_chain` (retriever + stuff-docs chain)  
   - This replaces the deprecated `RetrievalQA` and gives you full control.

4. **Prompt engineering is critical for citations**  
   - The model won’t cite sources unless you explicitly tell it to in the system prompt.  
   - Metadata (page number, source file) must be present in each document chunk.

5. **Chunking affects citation accuracy**  
   - `chunk_size=1000`, `chunk_overlap=200` is a good baseline.  
   - If a chunk spans multiple pages, page numbers become ambiguous (fine-tune later).

6. **Source documents are returned separately**  
   - The `response["context"]` holds the retrieved chunks – use them to show citations without relying on the LLM to format them perfectly.

7. **Testing locally is fast**  
   - Chroma runs in‑memory (or persists to disk). You can iterate quickly before moving to production vector databases.

---

##  Mental model for RAG

> **User question** → **Retriever** (finds relevant chunks) → **Prompt** (injects chunks as context) → **LLM** → **Answer + citations**

---

##  What we can now build

- Company policy FAQ bot  
- Personal document assistant (resumes, research papers, manuals)  
- Customer support bot that answers from product documentation  
